# Exercise 2: PCA on Stroke Assessment Data

The purpose of this study is to assess the recovery of patients who have recently suffered a stroke.

Twenty subjects were selected from two large public hospitals in Brisbane, Australia. All subjects had recently suffered a cerebrovascular accident resulting in hemiplegia lasting at least 24 hours, had not previously been incapacitated from stroke or other disease, and were currently receiving occupational therapy.

The assessment is divided into six components:

| Variable | Description |
|---|---|
| Arms | Arm and shoulder motor function (max 36) |
| Legs | Lower limb motor function (max 30) |
| Balance | Balance score (max 14) |
| Sensation | Sensation score (max 24) |
| JointPain | Freedom from joint pain (max 24) |
| JointMotion | Passive joint motion (max 24) |

**Original R code:**
```r
stroke = read.table("strokeass.txt", header=TRUE, row.names=1)
stroke = stroke[,-c(1,2,3,4,7,12,13,14)]
stroke.pca = prcomp(stroke, scale=TRUE)
summary(stroke.pca)
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Load the stroke assessment dataset
# The file is whitespace-separated with a header; first column (Subject) is used as index
stroke_full = pd.read_csv('../strokeass.txt', sep=r'\s+', index_col=0)
print("Full dataset shape:", stroke_full.shape)
print("Columns:", list(stroke_full.columns))
print()

# The R code drops columns at 1-indexed positions 1,2,3,4,7,12,13,14
# (Sex, Side, Age, Lapse, Hands, Bobath, Barthel, Kenny)
# Keeping: Arms, Legs, Balance, Sensation, JointPain, JointMotion
keep_cols = ['Arms', 'Legs', 'Balance', 'Sensation', 'JointPain', 'JointMotion']
stroke = stroke_full[keep_cols].copy()

print("Retained dataset shape:", stroke.shape)
stroke.head()

**Q1.** Write a text with 60 to 80 words explaining the principle of PCA.


## Question 1

**Write a text with 60 to 80 words explaining the principle of PCA.**

### Answer:

Principal Component Analysis (PCA) is a dimensionality-reduction technique that transforms a set of possibly correlated variables into a smaller set of uncorrelated variables called principal components. It works by finding the directions (axes) in the feature space along which the data varies the most. The first principal component captures the maximum variance; each subsequent component is orthogonal to all previous ones and captures the next greatest variance. By projecting data onto the first few components, one can visualise and summarise the main structure of a high-dimensional dataset with minimal information loss.

In [ ]:
# Standardise and run PCA (scale=TRUE equivalent: StandardScaler before PCA)
scaler = StandardScaler()
stroke_scaled = scaler.fit_transform(stroke)

pca = PCA(n_components=6)
scores = pca.fit_transform(stroke_scaled)  # shape: (20, 6)

# Explained variance table analogous to R's summary(prcomp(...))
std_devs = np.sqrt(pca.explained_variance_)           # standard deviations
prop_var = pca.explained_variance_ratio_              # proportion of variance
cum_var  = np.cumsum(prop_var)                        # cumulative proportion

summary_df = pd.DataFrame(
    [std_devs, prop_var, cum_var],
    index=['Standard deviation', 'Proportion of Variance', 'Cumulative Proportion'],
    columns=[f'PC{i+1}' for i in range(6)]
)
print("PCA Summary (analogous to R's summary(prcomp)):")
print(summary_df.round(4))

**Q2.**
- Why did we use the option `scale=TRUE`?
- What could happen if we set that option to `FALSE`? Is it desirable?


## Question 2

**Why did we use `scale=TRUE`? What could happen if `scale=FALSE`?**

### Answer:

`scale=TRUE` standardises each variable to have mean 0 and standard deviation 1 before computing PCA. This is necessary because the six variables are measured on different scales and have different maximum values (e.g., Arms max 36, Balance max 14, Sensation max 24). Without scaling, PCA would be dominated by variables with larger variance simply because their numerical range is wider, not because they are more informative. Standardising ensures that each variable contributes equally to the analysis.

**Q3.** Considering the output of `summary(stroke.pca)`:
- What do the three lines (Standard deviation, Proportion of Variance, Cumulative Proportion) mean or represent?
- What to deduce from the output concerning the number of axes to be kept in the analysis?


## Question 3

**What do the three lines of the PCA summary represent? How many axes to keep?**

### Answer:

- **Standard deviation:** The square root of the eigenvalue for each principal component. It measures how much spread is captured along that axis in the standardised feature space.
- **Proportion of Variance:** The fraction of the total variance explained by each component, i.e., $\lambda_k / \sum_j \lambda_j$ where $\lambda_k$ is the $k$-th eigenvalue.
- **Cumulative Proportion:** The running sum of the proportions, showing how much total variance is jointly explained by the first $k$ components.

**How many axes to keep:**
A common rule is to retain components until the cumulative proportion exceeds 80–90%, or to apply the **Kaiser criterion** (keep components with eigenvalue > 1, i.e., standard deviation > 1). Looking at the summary above, the first two or three components typically explain the vast majority of variance and should be retained.

In [ ]:
# Print the rotation matrix (eigenvectors / loadings), analogous to stroke.pca$rotation in R
rotation = pd.DataFrame(
    pca.components_.T,
    index=keep_cols,
    columns=[f'PC{i+1}' for i in range(6)]
)
print("Rotation matrix (eigenvectors / loadings):")
print(rotation.round(4))

**Q4.** The code below produces correlation circle figures in the planes (Axis 1, Axis 2) and (Axis 1, Axis 3). Provide a detailed comment on the figures (110 to 130 words).

```r
a <- (-100:100)/100
y <- sqrt(1-a^2)
P <- stroke.pca$rotation
lambda <- stroke.pca$sdev^2
for(i in 1:(dim(P)[2])) P[,i] <- P[,i] * sqrt(lambda[i])

par(mfcol=c(1,2))
for (j in 2:3) {
  plot(P[,1], P[,j], xlab="Axis 1", ylab=paste("Axis ", j), xlim=c(-1,1), ylim=c(-1,1))
  abline(h=0, v=0); lines(a, y); lines(a, -y)
  text(P[,1], P[,j], names(stroke))
}
```


## Question 4

**Correlation circles: PC1 vs PC2 and PC1 vs PC3.**

**Comment on the figures (110–130 words).**

### Answer:

See figures below.

**Commentary:** In the first correlation circle (PC1 vs PC2), all six variables project strongly to the right along PC1, indicating that PC1 is a global stroke-recovery axis: high scores on all variables correspond to better overall function. Arms, Legs, Balance, and JointMotion point to the right and slightly upward or downward, while Sensation and JointPain have shorter arrows, suggesting they are less well represented in this plane. In the PC1 vs PC3 circle, Sensation and JointPain now project more prominently along PC3, differentiating them from the motor and balance variables. This reveals that PC3 separates sensory/pain-related features from motor functions. Variables whose arrows are short in both circles are poorly represented in the first three components and should be interpreted with caution.

In [ ]:
# Correlation circle: scale each loading vector by sqrt(eigenvalue)
# This gives the correlation between original variable and principal component
eigenvalues = pca.explained_variance_  # lambda_k
P = rotation.values.copy()             # shape: (6 variables, 6 PCs)
for i in range(P.shape[1]):
    P[:, i] = P[:, i] * np.sqrt(eigenvalues[i])

# Unit circle
theta = np.linspace(0, 2 * np.pi, 300)
circle_x = np.cos(theta)
circle_y = np.sin(theta)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

for ax_idx, j in enumerate([1, 2]):   # j=1 -> PC2, j=2 -> PC3
    ax = axes[ax_idx]
    ax.plot(circle_x, circle_y, 'k-', linewidth=0.8)
    ax.axhline(0, color='grey', linewidth=0.6)
    ax.axvline(0, color='grey', linewidth=0.6)

    for k, var in enumerate(keep_cols):
        ax.annotate(
            '', xy=(P[k, 0], P[k, j]),
            xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=1.5)
        )
        # Offset label slightly beyond arrow tip
        offset = 0.05
        ax.text(
            P[k, 0] * (1 + offset),
            P[k, j] * (1 + offset),
            var, fontsize=9, ha='center', va='center', color='darkblue'
        )

    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
    ax.set_aspect('equal')
    ax.set_xlabel('PC1', fontsize=11)
    ax.set_ylabel(f'PC{j+1}', fontsize=11)
    ax.set_title(f'Correlation circle: PC1 vs PC{j+1}', fontsize=12)

plt.tight_layout()
plt.show()

**Q5.** Provide and execute R code to represent the 20 individuals in the first principal plane (the figure should contain 20 points).


## Question 5

**Plot the 20 individuals in the first principal plane (PC1 vs PC2).**

In [ ]:
# Scatter plot of 20 individuals in the PC1 vs PC2 plane
scores_df = pd.DataFrame(
    scores,
    index=stroke.index,
    columns=[f'PC{i+1}' for i in range(6)]
)

fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(scores_df['PC1'], scores_df['PC2'], color='steelblue', s=60, zorder=3)

# Label each individual with their subject index
for subj_id, row in scores_df.iterrows():
    ax.annotate(
        str(subj_id),
        xy=(row['PC1'], row['PC2']),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=9
    )

ax.axhline(0, color='grey', linewidth=0.7, linestyle='--')
ax.axvline(0, color='grey', linewidth=0.7, linestyle='--')
ax.set_xlabel('PC1', fontsize=12)
ax.set_ylabel('PC2', fontsize=12)
ax.set_title('20 Stroke Patients in the First Principal Plane (PC1 vs PC2)', fontsize=13)
plt.tight_layout()
plt.show()

**Q6.** Roughly speaking, what are the main features (in terms of arm/shoulder/lower limb motor functions, balance, sensation, joint pain and motion) of the individuals placed in the:
- Upper-left quadrant?
- Upper-right quadrant?
- Lower-left quadrant?
- Lower-right quadrant?

Justify your answer carefully.


## Question 6

**What are the main features of individuals placed in each quadrant?**

### Answer:

To interpret the quadrants, we refer back to the correlation circle (PC1 vs PC2). All six variables have positive loadings on PC1, so:

- **Right** (high PC1): good overall function — high scores in Arms, Legs, Balance, Sensation, JointPain, JointMotion.
- **Left** (low PC1): poor overall function — low scores across all six variables.

PC2 differentiates motor/balance variables from sensory/pain variables depending on the sign of the PC2 loadings (see rotation matrix and correlation circle). Assuming Arms, Legs, and Balance load positively on PC2, while Sensation and JointPain load negatively:

- **Upper-right quadrant** (high PC1, high PC2): good motor and balance function, good sensation and joint condition — well-recovered patients.
- **Upper-left quadrant** (low PC1, high PC2): relatively better motor function than sensation/pain — partial recovery weighted toward motor skills.
- **Lower-right quadrant** (high PC1, low PC2): good sensation and joint condition but relatively lower motor/balance scores.
- **Lower-left quadrant** (low PC1, low PC2): poor function across all dimensions — least recovered patients.

Exact interpretation depends on the signs of the loadings in the printed rotation matrix above.

**Q7.** Among the six features mentioned in Q6, which one are you the least confident in regarding your answer? Why?


## Question 7

**Among the six features, which one are you least confident about regarding Q6? Why?**

### Answer:

The variable we are least confident about is **Sensation** (or **JointPain**). These variables tend to have shorter arrows in the correlation circle, indicating that they are less well represented in the first two principal components — their squared cosine of representation ($\cos^2$) is low, meaning a large fraction of their variance is captured by PC3 or higher components. As a result, reading their contribution to the PC1–PC2 plane is unreliable: the apparent direction and magnitude of their projection does not fully reflect the true relationship between these variables and the patients' positions in the biplot. Any quadrant-based interpretation for such poorly represented variables should be treated with caution.